# Retrieval Component

## 1. Introduction

### 1.1. Install and Import Libraries

In [ ]:
# %pip install -U pandas tqdm python-dotenv neo4j qdrant-client

# %pip install --force-reinstall --no-deps "transformers==4.51.3" "huggingface_hub==0.30.2" "tokenizers==0.21.1"
# %pip install --upgrade --no-deps FlagEmbedding
# %pip install --force-reinstall "numpy==2.2.6"

2.2.6 4.51.3 0.30.2 0.21.1


In [15]:
# ============================================
# 1. Standard Library
# ============================================
import gc
import json
import math
import os
import re
import time
from collections import Counter, defaultdict
from dataclasses import dataclass
from importlib import reload
from pathlib import Path
from typing import Any, Iterable, Mapping

# ============================================
# 2. Third-party Libraries
# ============================================
import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from neo4j import GraphDatabase
from qdrant_client import QdrantClient
from qdrant_client import models as qdrant_models
from tqdm.auto import tqdm

def l2_normalize_matrix(values: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """L2-normalize rows without depending on scikit-learn."""
    array = np.asarray(values, dtype=np.float32)
    norms = np.linalg.norm(array, axis=1, keepdims=True)
    return array / np.maximum(norms, eps)

# ============================================
# 3. Project Helpers
# ============================================
# Keep startup imports light. BGE-M3 and LLM helpers import Transformers, which can
# trip over local binary package issues; load them lazily in the cells that need them.
import src.graph_kb_neo4j as neo4j_helpers
import src.retrieval_drivers as retrieval_driver_helpers
import src.retrieval_sparse as sparse_retrieval_helpers
import src.retrieval_query_generation as query_generation_helpers

reload(neo4j_helpers)
reload(retrieval_driver_helpers)
reload(sparse_retrieval_helpers)
reload(query_generation_helpers)

from src.retrieval_query_generation import load_generated_qrels, read_jsonl
from src.retrieval_drivers import Neo4jGraphRetrievalDriver, QdrantRetrievalDriver
from src.retrieval_sparse import QdrantSparseRetriever

import src.bge_m3_embedding as bge_m3_helpers
reload(bge_m3_helpers)
from src.bge_m3_embedding import BGEM3Embedder

import src.retrieval_llms as retrieval_llm_helpers
reload(retrieval_llm_helpers)
from src.retrieval_llms import RelevanceJudgeLLM, extract_first_json_object


### 1.2. Environment Setup & Service Initialization

#### 1.2.1. Google Colab Setup

Run this section if you are running this notebook on Google Colab. Otherwise, run section 1.3.2.

In [ ]:
# from google.colab import userdata

In [ ]:
# qdrant_retrieval_driver = QdrantRetrievalDriver.from_environment(
#     env_path=".env",
#     use_colab_userdata=True,
#     default_collection_name="archive-chunks",
# )
# COLLECTION_NAME = qdrant_retrieval_driver.collection_name
#
# print(qdrant_retrieval_driver.verify_connectivity())
# print(f"Qdrant collection: {COLLECTION_NAME}")

In [ ]:
# neo4j_graph_driver = Neo4jGraphRetrievalDriver.from_environment(
#     env_path=".env",
#     use_colab_userdata=True,
# )
# neo4j_graph_driver.verify_connectivity()
# print("Neo4j graph retrieval driver connected.")

#### 1.2.2. Local Setup

In [2]:
qdrant_retrieval_driver = QdrantRetrievalDriver.from_environment(
    env_path=".env",
    use_colab_userdata=True,
    default_collection_name="archive-chunks",
)
COLLECTION_NAME = qdrant_retrieval_driver.collection_name

print(qdrant_retrieval_driver.verify_connectivity())
print(f"Qdrant collection: {COLLECTION_NAME}")

collections=[CollectionDescription(name='archive-chunks')]
Qdrant collection: archive-chunks


In [3]:
neo4j_graph_driver = Neo4jGraphRetrievalDriver.from_environment(
    env_path=".env",
    use_colab_userdata=True,
)
neo4j_graph_driver.verify_connectivity()
print("Neo4j graph retrieval driver connected.")

Neo4j graph retrieval driver connected.


### 1.3. Initialize LLM Services

In [ ]:
INITIALIZE_RELEVANCE_JUDGE_LLM = True

if INITIALIZE_RELEVANCE_JUDGE_LLM:
    relevance_judge_llm = RelevanceJudgeLLM(
        token=HUGGING_FACE_TOKEN,
        torch_dtype=LLM_TORCH_DTYPE,
        load_in_4bit=LLM_LOAD_IN_4BIT,
    )
    print(f"Loaded relevance-judge LLM on: {relevance_judge_llm.device}")
else:
    relevance_judge_llm = None
    print("Relevance judge initialization skipped for now.")

## 2. Tune and Smoke Test Retrieval Strategies

In [16]:
RETRIEVAL_EXPORT_DIR = Path("data/evaluation/retrieval_runs")
RETRIEVAL_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

ARCHIVE_CHUNKS_CSV = Path("data/graph_kb_exports/step_01_archive_skeleton/chunks.csv")
RETRIEVAL_SMOKE_TEST_QUERY_ID = "smoke_query_001"
RETRIEVAL_SMOKE_TEST_QUERY = "What concern did legal scholars raise about CIA drone strikes?"
RETRIEVAL_SMOKE_TEST_TOP_K = 10

print(RETRIEVAL_SMOKE_TEST_QUERY)

What concern did legal scholars raise about CIA drone strikes?


### 2.1. Sparse Retrieval

#### 2.1.1. Tune top-k

In [6]:
from importlib import reload

import src.retrieval_sparse as sparse_retrieval_helpers
reload(sparse_retrieval_helpers)

from src.retrieval_sparse import QdrantSparseRetriever

In [7]:
# TODO: tune sparse retrieval top-k after generated qrels are ready.
# This cell will evaluate candidate top-k values using only query_id/query for retrieval
# and qrel fields only after retrieval for evaluation.
SPARSE_RETRIEVAL_TOP_K_CANDIDATES = [5, 10, 20, 50]
BEST_SPARSE_RETRIEVAL_TOP_K = RETRIEVAL_SMOKE_TEST_TOP_K
print(f"Using smoke-test sparse top-k: {BEST_SPARSE_RETRIEVAL_TOP_K}")

Using smoke-test sparse top-k: 10


#### 2.1.2. Smoke test

In [20]:
SPARSE_VECTOR_NAME = "sparse"
SPARSE_QUERY_EMBEDDING_MAX_LENGTH = 1024

if "qdrant_retrieval_driver" not in globals():
    raise RuntimeError("Initialize qdrant_retrieval_driver in section 1.2.2 first.")

if "sparse_query_embedder" not in globals():
    # Lazy import keeps the notebook startup light while other long-running jobs are active.
    from src.bge_m3_embedding import BGEM3Embedder
    sparse_query_embedder = BGEM3Embedder(
        device="cuda" if torch.cuda.is_available() else "cpu",
        require_cuda=False,
    )
    print(f"Loaded sparse query embedder on: {sparse_query_embedder.device}")

if "qdrant_sparse_retriever" not in globals():
    qdrant_sparse_retriever = QdrantSparseRetriever(
        qdrant_driver=qdrant_retrieval_driver,
        embedder=sparse_query_embedder,
        vector_name=SPARSE_VECTOR_NAME,
        embedding_max_length=SPARSE_QUERY_EMBEDDING_MAX_LENGTH,
    )

sparse_smoke_results_df = qdrant_sparse_retriever.retrieve(
    RETRIEVAL_SMOKE_TEST_QUERY,
    query_id=RETRIEVAL_SMOKE_TEST_QUERY_ID,
    top_k=BEST_SPARSE_RETRIEVAL_TOP_K,
)

display(sparse_smoke_results_df[[
    "rank",
    "score",
    "chunk_id",
    "dataset",
    "modality",
    "title",
    "retrieval_text_preview",
]])

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 16.67it/s]


,rank,score,chunk_id,dataset,modality,title,retrieval_text_preview
0,1,0.226746,e6be75d4356e2c23cc1e1370,cnn_dailymail,article,CNN/DailyMail Article 0348c10a8212,Dataset: cnn_dailymail\nModality: article\nTit...
1,2,0.221275,84826471808b8a1a3335a73b,cnn_dailymail,article,CNN/DailyMail Article 0348c10a8212,Dataset: cnn_dailymail\nModality: article\nTit...
2,3,0.216745,cda827e281f3e728d26877de,cnn_dailymail,article,CNN/DailyMail Article 0348c10a8212,Dataset: cnn_dailymail\nModality: article\nTit...
3,4,0.214957,940efcd4a433ba1f46ccdb58,cnn_dailymail,article,CNN/DailyMail Article 0348c10a8212,Dataset: cnn_dailymail\nModality: article\nTit...
4,5,0.209537,be839e1bf2c072527ae0b4c7,cnn_dailymail,article,CNN/DailyMail Article 2fe159232cbb,Dataset: cnn_dailymail\nModality: article\nTit...
5,6,0.207418,32e749047c6217008d8350ab,cnn_dailymail,article,CNN/DailyMail Article 2cdbffd07118,Dataset: cnn_dailymail\nModality: article\nTit...
6,7,0.198366,6c0fbfe5017a537cb67c7f15,cnn_dailymail,article,CNN/DailyMail Article 2cdbffd07118,Dataset: cnn_dailymail\nModality: article\nTit...
7,8,0.197307,aa35e6b5b8815b5f85f9c3c0,cnn_dailymail,article,CNN/DailyMail Article 0348c10a8212,Dataset: cnn_dailymail\nModality: article\nTit...
8,9,0.195189,e98de4a12c1a53b779ff457f,cnn_dailymail,article,CNN/DailyMail Article 8efbf25de097,Dataset: cnn_dailymail\nModality: article\nTit...
9,10,0.194322,f3ec814fae842b9d034822ed,cnn_dailymail,article,CNN/DailyMail Article 9f3364a67e11,Dataset: cnn_dailymail\nModality: article\nTit...


In [23]:
# Inspect the full text for the top sparse retrieval results.
SPARSE_SMOKE_PREVIEW_TOP_N = 3

if "sparse_smoke_results_df" not in globals() or sparse_smoke_results_df.empty:
    raise RuntimeError("Run the sparse smoke-test retrieval cell first.")

for _, row in sparse_smoke_results_df.head(SPARSE_SMOKE_PREVIEW_TOP_N).iterrows():
    print("=" * 100)
    print(f"Rank {int(row['rank'])} | score={row['score']:.4f} | chunk_id={row['chunk_id']}")
    print(f"Dataset={row['dataset']} | Modality={row['modality']} | Document={row['document_id']}")
    print(f"Title: {row['title']}")
    print("-" * 100)
    print(str(row.get("retrieval_text", ""))[:3000])
    print()

Rank 1 | score=0.2267 | chunk_id=e6be75d4356e2c23cc1e1370
Dataset=cnn_dailymail | Modality=article | Document=2dd89e248d507aacb58404d3
Title: CNN/DailyMail Article 0348c10a8212
----------------------------------------------------------------------------------------------------
Dataset: cnn_dailymail
Modality: article
Title: CNN/DailyMail Article 0348c10a8212
Summary: NEW: ACLU calls drone attacks part of illegal program for U.S. to target, kill terror suspects .
Since President Obama took office, number of drone attacks has risen .
U.S. law professors debate legality of such attacks during a House subcommittee hearing .
Biggest controversy: legality of strikes conducted by CIA, as opposed to U.S. military .
Metadata: {"hf_dataset": "abisee/cnn_dailymail", "hf_config": "3.0.0", "hf_split": "train", "original_fields": ["article", "highlights", "id"]}
Content: John Tierney, D-Massachusetts, the subcommittee's chairman. "Our interpretation of how these standards apply to the use of unmanne

### 2.2. Dense Retrieval

#### 2.2.1. Tune top-k

In [17]:
from importlib import reload

import src.retrieval_dense as dense_retrieval_helpers
reload(dense_retrieval_helpers)

from src.retrieval_dense import QdrantDenseRetriever

In [18]:
# TODO: tune dense retrieval top-k after generated qrels are ready.
# This cell will evaluate candidate top-k values using only query_id/query for retrieval
# and qrel fields only after retrieval for evaluation.
DENSE_RETRIEVAL_TOP_K_CANDIDATES = [5, 10, 20, 50]
BEST_DENSE_RETRIEVAL_TOP_K = RETRIEVAL_SMOKE_TEST_TOP_K
print(f"Using smoke-test dense top-k: {BEST_DENSE_RETRIEVAL_TOP_K}")

Using smoke-test dense top-k: 10


#### 2.2.2. Smoke test

In [21]:
DENSE_VECTOR_NAME = "dense"
DENSE_QUERY_EMBEDDING_MAX_LENGTH = 1024

if "qdrant_retrieval_driver" not in globals():
    raise RuntimeError("Initialize qdrant_retrieval_driver in section 1.2.2 first.")

if "bge_m3_query_embedder" not in globals():
    if "sparse_query_embedder" in globals():
        bge_m3_query_embedder = sparse_query_embedder
        print(f"Reusing BGE-M3 query embedder on: {bge_m3_query_embedder.device}")
    else:
        from src.bge_m3_embedding import BGEM3Embedder
        bge_m3_query_embedder = BGEM3Embedder(
            device="cuda" if torch.cuda.is_available() else "cpu",
            require_cuda=False,
        )
        print(f"Loaded BGE-M3 query embedder on: {bge_m3_query_embedder.device}")

if "qdrant_dense_retriever" not in globals():
    qdrant_dense_retriever = QdrantDenseRetriever(
        qdrant_driver=qdrant_retrieval_driver,
        embedder=bge_m3_query_embedder,
        vector_name=DENSE_VECTOR_NAME,
        embedding_max_length=DENSE_QUERY_EMBEDDING_MAX_LENGTH,
    )

dense_smoke_results_df = qdrant_dense_retriever.retrieve(
    RETRIEVAL_SMOKE_TEST_QUERY,
    query_id=RETRIEVAL_SMOKE_TEST_QUERY_ID,
    top_k=BEST_DENSE_RETRIEVAL_TOP_K,
)

display(dense_smoke_results_df[[
    "rank",
    "score",
    "chunk_id",
    "dataset",
    "modality",
    "title",
    "retrieval_text_preview",
]])

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 16.95it/s]


,rank,score,chunk_id,dataset,modality,title,retrieval_text_preview
0,1,0.734324,84826471808b8a1a3335a73b,cnn_dailymail,article,CNN/DailyMail Article 0348c10a8212,Dataset: cnn_dailymail\nModality: article\nTit...
1,2,0.725239,e6be75d4356e2c23cc1e1370,cnn_dailymail,article,CNN/DailyMail Article 0348c10a8212,Dataset: cnn_dailymail\nModality: article\nTit...
2,3,0.700272,cda827e281f3e728d26877de,cnn_dailymail,article,CNN/DailyMail Article 0348c10a8212,Dataset: cnn_dailymail\nModality: article\nTit...
3,4,0.691237,940efcd4a433ba1f46ccdb58,cnn_dailymail,article,CNN/DailyMail Article 0348c10a8212,Dataset: cnn_dailymail\nModality: article\nTit...
4,5,0.687136,aa35e6b5b8815b5f85f9c3c0,cnn_dailymail,article,CNN/DailyMail Article 0348c10a8212,Dataset: cnn_dailymail\nModality: article\nTit...
5,6,0.661912,be839e1bf2c072527ae0b4c7,cnn_dailymail,article,CNN/DailyMail Article 2fe159232cbb,Dataset: cnn_dailymail\nModality: article\nTit...
6,7,0.656365,67e9541f255761fe88837d03,cnn_dailymail,article,CNN/DailyMail Article 2fe159232cbb,Dataset: cnn_dailymail\nModality: article\nTit...
7,8,0.652834,6c0fbfe5017a537cb67c7f15,cnn_dailymail,article,CNN/DailyMail Article 2cdbffd07118,Dataset: cnn_dailymail\nModality: article\nTit...
8,9,0.651553,32e749047c6217008d8350ab,cnn_dailymail,article,CNN/DailyMail Article 2cdbffd07118,Dataset: cnn_dailymail\nModality: article\nTit...
9,10,0.645025,6b0dcb9d2613e228fe7807e1,cnn_dailymail,article,CNN/DailyMail Article 8efbf25de097,Dataset: cnn_dailymail\nModality: article\nTit...


In [22]:
# Inspect the full text for the top dense retrieval results.
DENSE_SMOKE_PREVIEW_TOP_N = 3

if "dense_smoke_results_df" not in globals() or dense_smoke_results_df.empty:
    raise RuntimeError("Run the dense smoke-test retrieval cell first.")

for _, row in dense_smoke_results_df.head(DENSE_SMOKE_PREVIEW_TOP_N).iterrows():
    print("=" * 100)
    print(f"Rank {int(row['rank'])} | score={row['score']:.4f} | chunk_id={row['chunk_id']}")
    print(f"Dataset={row['dataset']} | Modality={row['modality']} | Document={row['document_id']}")
    print(f"Title: {row['title']}")
    print("-" * 100)
    print(str(row.get("retrieval_text", ""))[:3000])
    print()

Rank 1 | score=0.7343 | chunk_id=84826471808b8a1a3335a73b
Dataset=cnn_dailymail | Modality=article | Document=2dd89e248d507aacb58404d3
Title: CNN/DailyMail Article 0348c10a8212
----------------------------------------------------------------------------------------------------
Dataset: cnn_dailymail
Modality: article
Title: CNN/DailyMail Article 0348c10a8212
Summary: NEW: ACLU calls drone attacks part of illegal program for U.S. to target, kill terror suspects .
Since President Obama took office, number of drone attacks has risen .
U.S. law professors debate legality of such attacks during a House subcommittee hearing .
Biggest controversy: legality of strikes conducted by CIA, as opposed to U.S. military .
Metadata: {"hf_dataset": "abisee/cnn_dailymail", "hf_config": "3.0.0", "hf_split": "train", "original_fields": ["article", "highlights", "id"]}
Content: And most importantly, they are not trained in the law of armed conflict." O'Connell also said that "we know from empirical data ..

### 2.3. Hybrid Dense-Sparse Retrieval

#### 2.3.1. Tune top-k

In [29]:
from importlib import reload

import src.retrieval_hybrid as hybrid_retrieval_helpers
reload(hybrid_retrieval_helpers)

from src.retrieval_hybrid import QdrantHybridRrfRetriever

In [30]:
# TODO: tune hybrid RRF top-k and prefetch limits after generated qrels are ready.
# Qdrant hybrid retrieval uses server-side RRF fusion over named dense and sparse vectors.
HYBRID_RETRIEVAL_TOP_K_CANDIDATES = [5, 10, 20, 50]
HYBRID_PREFETCH_MULTIPLIER_CANDIDATES = [3, 5, 10]
BEST_HYBRID_RETRIEVAL_TOP_K = RETRIEVAL_SMOKE_TEST_TOP_K
BEST_HYBRID_PREFETCH_MULTIPLIER = 5
print(f"Using smoke-test hybrid top-k: {BEST_HYBRID_RETRIEVAL_TOP_K}")
print(f"Using smoke-test hybrid prefetch multiplier: {BEST_HYBRID_PREFETCH_MULTIPLIER}")

Using smoke-test hybrid top-k: 10
Using smoke-test hybrid prefetch multiplier: 5


#### 2.3.2. Smoke test

In [33]:
HYBRID_DENSE_VECTOR_NAME = "dense"
HYBRID_SPARSE_VECTOR_NAME = "sparse"
HYBRID_QUERY_EMBEDDING_MAX_LENGTH = 1024

if "qdrant_retrieval_driver" not in globals():
    raise RuntimeError("Initialize qdrant_retrieval_driver in section 1.2.2 first.")

if "bge_m3_query_embedder" not in globals():
    if "sparse_query_embedder" in globals():
        bge_m3_query_embedder = sparse_query_embedder
        print(f"Reusing BGE-M3 query embedder on: {bge_m3_query_embedder.device}")
    else:
        from src.bge_m3_embedding import BGEM3Embedder
        bge_m3_query_embedder = BGEM3Embedder(
            device="cuda" if torch.cuda.is_available() else "cpu",
            require_cuda=False,
        )
        print(f"Loaded BGE-M3 query embedder on: {bge_m3_query_embedder.device}")

if "qdrant_hybrid_retriever" not in globals():
    qdrant_hybrid_retriever = QdrantHybridRrfRetriever(
        qdrant_driver=qdrant_retrieval_driver,
        embedder=bge_m3_query_embedder,
        dense_vector_name=HYBRID_DENSE_VECTOR_NAME,
        sparse_vector_name=HYBRID_SPARSE_VECTOR_NAME,
        embedding_max_length=HYBRID_QUERY_EMBEDDING_MAX_LENGTH,
        prefetch_multiplier=BEST_HYBRID_PREFETCH_MULTIPLIER,
    )

hybrid_smoke_results_df = qdrant_hybrid_retriever.retrieve(
    RETRIEVAL_SMOKE_TEST_QUERY,
    query_id=RETRIEVAL_SMOKE_TEST_QUERY_ID,
    top_k=BEST_HYBRID_RETRIEVAL_TOP_K,
)

display(hybrid_smoke_results_df[[
    "rank",
    "score",
    "chunk_id",
    "dataset",
    "modality",
    "title",
    "retrieval_text_preview",
]])

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 18.68it/s]


,rank,score,chunk_id,dataset,modality,title,retrieval_text_preview
0,1,0.833333,84826471808b8a1a3335a73b,cnn_dailymail,article,CNN/DailyMail Article 0348c10a8212,Dataset: cnn_dailymail\nModality: article\nTit...
1,2,0.833333,e6be75d4356e2c23cc1e1370,cnn_dailymail,article,CNN/DailyMail Article 0348c10a8212,Dataset: cnn_dailymail\nModality: article\nTit...
2,3,0.500000,cda827e281f3e728d26877de,cnn_dailymail,article,CNN/DailyMail Article 0348c10a8212,Dataset: cnn_dailymail\nModality: article\nTit...
3,4,0.400000,940efcd4a433ba1f46ccdb58,cnn_dailymail,article,CNN/DailyMail Article 0348c10a8212,Dataset: cnn_dailymail\nModality: article\nTit...
4,5,0.309524,be839e1bf2c072527ae0b4c7,cnn_dailymail,article,CNN/DailyMail Article 2fe159232cbb,Dataset: cnn_dailymail\nModality: article\nTit...
5,6,0.277778,aa35e6b5b8815b5f85f9c3c0,cnn_dailymail,article,CNN/DailyMail Article 0348c10a8212,Dataset: cnn_dailymail\nModality: article\nTit...
6,7,0.242857,32e749047c6217008d8350ab,cnn_dailymail,article,CNN/DailyMail Article 2cdbffd07118,Dataset: cnn_dailymail\nModality: article\nTit...
7,8,0.236111,6c0fbfe5017a537cb67c7f15,cnn_dailymail,article,CNN/DailyMail Article 2cdbffd07118,Dataset: cnn_dailymail\nModality: article\nTit...
8,9,0.167832,6b0dcb9d2613e228fe7807e1,cnn_dailymail,article,CNN/DailyMail Article 8efbf25de097,Dataset: cnn_dailymail\nModality: article\nTit...
9,10,0.156250,67e9541f255761fe88837d03,cnn_dailymail,article,CNN/DailyMail Article 2fe159232cbb,Dataset: cnn_dailymail\nModality: article\nTit...


In [34]:
# Inspect the full text for the top hybrid retrieval results.
HYBRID_SMOKE_PREVIEW_TOP_N = 3

if "hybrid_smoke_results_df" not in globals() or hybrid_smoke_results_df.empty:
    raise RuntimeError("Run the hybrid smoke-test retrieval cell first.")

for _, row in hybrid_smoke_results_df.head(HYBRID_SMOKE_PREVIEW_TOP_N).iterrows():
    print("=" * 100)
    print(f"Rank {int(row['rank'])} | score={row['score']:.4f} | chunk_id={row['chunk_id']}")
    print(f"Dataset={row['dataset']} | Modality={row['modality']} | Document={row['document_id']}")
    print(f"Title: {row['title']}")
    print("-" * 100)
    print(str(row.get("retrieval_text", ""))[:3000])
    print()

Rank 1 | score=0.8333 | chunk_id=84826471808b8a1a3335a73b
Dataset=cnn_dailymail | Modality=article | Document=2dd89e248d507aacb58404d3
Title: CNN/DailyMail Article 0348c10a8212
----------------------------------------------------------------------------------------------------
Dataset: cnn_dailymail
Modality: article
Title: CNN/DailyMail Article 0348c10a8212
Summary: NEW: ACLU calls drone attacks part of illegal program for U.S. to target, kill terror suspects .
Since President Obama took office, number of drone attacks has risen .
U.S. law professors debate legality of such attacks during a House subcommittee hearing .
Biggest controversy: legality of strikes conducted by CIA, as opposed to U.S. military .
Metadata: {"hf_dataset": "abisee/cnn_dailymail", "hf_config": "3.0.0", "hf_split": "train", "original_fields": ["article", "highlights", "id"]}
Content: And most importantly, they are not trained in the law of armed conflict." O'Connell also said that "we know from empirical data ..